# Ensemble Retriever — 하이브리드 검색

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Sparse(BM25)**와 **Dense(벡터)** 검색 방식의 차이와 각각의 강점 이해하기
- `EnsembleRetriever`로 두 방식을 결합하는 방법 익히기
- `weights` 파라미터로 가중치를 조정해 검색 성능 튜닝하기

## 사전 지식

- BM25: 키워드 빈도를 활용한 전통적 검색 방식
- Dense Retrieval: 임베딩 벡터 간 유사도 기반 검색

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> B[BM25<br/>키워드 검색]:::process
    Q --> V[FAISS<br/>의미 검색]:::process
    B -- 가중치 0.5 --> E[EnsembleRetriever<br/>RRF 알고리즘]:::process
    V -- 가중치 0.5 --> E
    E --> R[최적 검색 결과]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## 핵심 아이디어

| 특성 | Sparse / BM25 | Dense / Vector |
|------|:---:|:---:|
| 기반 | 키워드 매칭 | 의미 유사도 |
| 강점 | 정확한 용어 매칭 | 문맥·동의어 이해 |
| 약점 | 동의어 못 찾음 | 키워드 민감도 낮음 |

두 방식을 결합하면 각각의 약점을 보완해요. **RRF(Reciprocal Rank Fusion)** 알고리즘으로 결과를 통합합니다:

```
score = Σ(weight_i / (k + rank_i))
```

## 환경 설정

In [1]:
from dotenv import load_dotenv

load_dotenv()


True

## 1. 문서 준비 및 개별 Retriever 생성

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ---------------------------------------------------
# 하이브리드 검색을 위한 문서 준비
# ---------------------------------------------------

# ============================================================
# TODO: ai-story.txt를 로드하고 500자 청크로 분할하세요
# 힌트: TextLoader → load() → RecursiveCharacterTextSplitter → split_documents()
# 예상 결과: 분할된 청크 수 출력
# ============================================================

# 문서 로드 및 분할
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(documents)



In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# 1. BM25 Retriever (Sparse - 키워드 기반)
bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 3

# 2. FAISS Retriever (Dense - 의미 기반)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [5]:
# ---------------------------------------------------
# 개별 Retriever로 동일 쿼리를 검색하여 차이점 확인
# ---------------------------------------------------

# ============================================================
# TODO: bm25_retriever와 faiss_retriever에 동일한 쿼리를 각각 실행하세요
# 힌트: retriever.invoke(query) 후 결과 출력
# 예상 결과: BM25는 키워드 매칭, FAISS는 의미 유사도 기반 문서 반환
# ============================================================

# 개별 retriever 성능 비교
query = "딥러닝 모델에서 가중치 학습은 어떻게 이뤄지나요?"

bm25_docs = bm25_retriever.invoke(query)
faiss_docs = faiss_retriever.invoke(query)

print(f":memo: 쿼리: {query}\n")
print("="*80)
print("[BM25 결과 - 키워드 기반]")
print("="*80)
for i, doc in enumerate(bm25_docs, 1):
    print(f"[{i}] {doc.page_content[:100]}...")
    print()

print("="*80)
print("[FAISS 결과 - 의미 기반]")
print("="*80)
for i, doc in enumerate(faiss_docs, 1):
    print(f"[{i}] {doc.page_content[:100]}...")

:memo: 쿼리: 딥러닝 모델에서 가중치 학습은 어떻게 이뤄지나요?

[BM25 결과 - 키워드 기반]
[1] 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Atte...

[2] Word2Vec의 벡터 표현은 다양한 NLP 작업에 활용될 수 있습니다. 예를 들어, 단어의 유사도 측정, 문장이나 문서의 벡터 표현 생성, 기계 번역, 감정 분석 등이 있습니다....

[3] Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이...

[FAISS 결과 - 의미 기반]
[1] 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Atte...
[2] Attention is all you need

"Attention Is All You Need"는 2017년에 발표된 논문으로, 변환기(Transformer) 모델의 아키텍처를 ...
[3] NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사...


## 2. EnsembleRetriever 생성 및 사용

두 retriever를 결합하여 각각의 강점을 활용합니다.

In [ ]:
from langchain.retrievers import EnsembleRetriever

# EnsembleRetriever 생성 (가중치 5:5)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5],
)

# Ensemble 검색
ensemble_docs = ensemble_retriever.invoke(query)

# 아래에 코드를 작성하세요


In [9]:
ensemble_docs

[Document(metadata={'source': './data/ai-story.txt'}, page_content="어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Attention)'은 입력 데이터 내의 각 요소가 서로 얼마나 관련이 있는지를 계산하여, 모델이 문맥을 더 잘 이해할 수 있게 합니다. 이는 특히 문장 내에서 단어 간의 관계를 파악하는 데 매우 효과적이며, 문장의 길이에 관계없이 일정한 연산 시간으로 처리할 수 있습니다."),
 Document(metadata={'source': './data/ai-story.txt'}, page_content='Word2Vec의 벡터 표현은 다양한 NLP 작업에 활용될 수 있습니다. 예를 들어, 단어의 유사도 측정, 문장이나 문서의 벡터 표현 생성, 기계 번역, 감정 분석 등이 있습니다. 또한, 벡터 연산을 통해 단어 간의 의미적 관계를 추론하는 것이 가능해집니다. 예를 들어, "king" - "man" + "woman"과 같은 벡터 연산을 수행하면, 결과적으로 "queen"과 유사한 벡터를 가진 단어를 찾을 수 있습니다.\n\nWord2Vec의 성공 이후, 이와 유사한 다른 단어 임베딩 기법들도 개발되었습니다. 그러나 Word2Vec은 그 간결함과 효율성, 높은 성능으로 인해 여전히 광범위하게 사용되며, NLP 분야에서 기본적인 도구로 자리 잡았습니다. Word2Vec는 단순한 텍스트 데이터를 통해 복잡한 언어의 의미 구조를 학습할 수 있는 강력한 방법을 제공함으로써, 컴퓨터가 인간 언어를 이해하는 방식을 혁신적으로 개선하였습니다.'),
 Document(id='ddfa5960-9293-4479-9c91-1eb22b9e709c', metadata={'source': './data/ai-story.txt'}, page_content='Attention is all you need\n\

In [8]:
# ---------------------------------------------------
# EnsembleRetriever의 가중치를 변경하여 결과 차이 실험
# ---------------------------------------------------

# ============================================================
# TODO: BM25 우선(7:3)과 FAISS 우선(3:7) 두 가지 가중치로 같은 쿼리를 검색하세요
# 힌트: EnsembleRetriever(retrievers=[...], weights=[w1, w2])
# 예상 결과: 가중치에 따라 상위 문서 순서가 달라짐
# ============================================================

# 가중치 실험

# 케이스 1: BM25 우선 (7:3)
# 전문 용어가 많거나 정확한 키워드 매칭이 중요할 때
print(f":memo: 쿼리: {query}\n")
print("="*80)
print("[Ensemble 결과 - 하이브리드 검색]")
print("="*80)
for i, doc in enumerate(ensemble_docs, 1):
    print(f"[{i}] {doc.page_content[:100]}...")
    print()

print("\n:bulb: 장점:")
print("  - BM25와 FAISS의 결과를 모두 반영")
print("  - RRF 알고리즘으로 최적 순위 결정")
print("  - 더 다양하고 관련성 높은 결과")
# 케이스 2: FAISS 우선 (3:7)
# 자연어 질문이 많거나 동의어를 고려해야 할 때
# 가중치 실험
test_query = "Transformer 아키텍처의 핵심 요소는?"

# 케이스 1: BM25 우선 (7:3)
# 전문 용어가 많거나 정확한 키워드 매칭이 중요할 때
ensemble_bm25_heavy = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.7, 0.3]
)

# 케이스 2: FAISS 우선 (3:7)
# 자연어 질문이 많거나 동의어를 고려해야 할 때
ensemble_faiss_heavy = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.3, 0.7]
)

docs_bm25_heavy = ensemble_bm25_heavy.invoke(test_query)
docs_faiss_heavy = ensemble_faiss_heavy.invoke(test_query)

print(f":memo: 쿼리: {test_query}\n")
print("="*80)
print("[BM25 우선 (7:3) - 키워드 중시]")
print("="*80)
for i, doc in enumerate(docs_bm25_heavy[:2], 1):
    print(f"[{i}] {doc.page_content[:80]}...")
print()

print("="*80)
print("[FAISS 우선 (3:7) - 의미 중시]")
print("="*80)
for i, doc in enumerate(docs_faiss_heavy[:2], 1):
    print(f"[{i}] {doc.page_content[:80]}...")
print()

:memo: 쿼리: 딥러닝 모델에서 가중치 학습은 어떻게 이뤄지나요?

[Ensemble 결과 - 하이브리드 검색]
[1] 어텐션 메커니즘은 입력 데이터의 서로 다른 부분에 대해 모델이 얼마나 많은 '주의'를 기울여야 하는지를 결정합니다. 특히, 변환기 모델에서 사용되는 '셀프 어텐션(Self-Atte...

[2] Word2Vec의 벡터 표현은 다양한 NLP 작업에 활용될 수 있습니다. 예를 들어, 단어의 유사도 측정, 문장이나 문서의 벡터 표현 생성, 기계 번역, 감정 분석 등이 있습니다....

[3] Attention is all you need

"Attention Is All You Need"는 2017년에 발표된 논문으로, 변환기(Transformer) 모델의 아키텍처를 ...

[4] Hugging Face의 Transformers 라이브러리는 다양한 NLP 작업에 활용될 수 있습니다. 이에는 텍스트 분류, 정보 추출, 질문 응답, 요약 생성, 텍스트 생성 등이...

[5] NLP의 핵심 과제 중 하나는 자연어의 모호성과 다양성을 처리하는 것입니다. 인간 언어는 복잡하고, 문맥에 따라 그 의미가 달라질 수 있으며, 같은 단어나 구문이 다양한 의미로 사...


:bulb: 장점:
  - BM25와 FAISS의 결과를 모두 반영
  - RRF 알고리즘으로 최적 순위 결정
  - 더 다양하고 관련성 높은 결과
:memo: 쿼리: Transformer 아키텍처의 핵심 요소는?

[BM25 우선 (7:3) - 키워드 중시]
[1] Scikit Learn

Scikit-learn은 Python 언어를 위한 또 다른 핵심 라이브러리로, 기계 학습의 다양한 알고리즘을 구현하기 ...
[2] 핵심 특징 중 하나는 다양한 기계 학습 모델을 일관된 인터페이스로 제공한다는 점입니다. 이는 사용자가 알고리즘 간에 쉽게 전환할 수 있게 하여,...

[FAISS 우선 (3:7) - 의미 중시]
[1] Attention is all you need

"Attention

## 마무리 정리

이 노트북에서 배운 내용을 정리해요:

| 가중치 설정 | 상황 | 예시 |
|-----------|------|------|
| `[0.7, 0.3]` | 전문 용어·고유명사 중심 | 의학, 법률 문서 |
| `[0.5, 0.5]` | 일반적 사용 (기본 권장) | 뉴스, 블로그 |
| `[0.3, 0.7]` | 자연어 질문·동의어 많음 | 고객 상담, Q&A |

> **실무 팁**: 가중치 최적값은 데이터와 쿼리 유형에 따라 달라집니다. 검색 품질을 평가할 수 있는 소규모 테스트 셋을 만들어 두고, 여러 가중치 조합을 비교하는 것이 좋아요.

### 다음 노트북 예고

**04-LongContextReorder**: LLM이 긴 컨텍스트의 중간 부분을 잘 활용하지 못하는 "Lost-in-the-Middle" 문제와 해결법을 배워요.